# Penguins.CSV Cleaning
Done by DE. Mohammad Maher

In [ ]:
import pandas as pd

In [ ]:
url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/master/inst/extdata/penguins.csv"

pd.read_csv(url).to_csv("data/penguins.csv", index=False)

In [ ]:
df = pd.read_csv('data/penguins.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

# Audit missing data
Report how many missing values exist in each column, and what percentage of rows are affected overall.

In [ ]:
nrows = df.isna().sum()
nrows

In [ ]:
missing_val_percent = (df.isna().mean() * 100).__round__(2)
missing_val_percent

# Fill numeric gaps
Fill missing bill_length_mm, bill_depth_mm, flipper_length_mm, and body_mass_g using the median value for that penguin's species (not the overall median).

In [ ]:
df.groupby('species')[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].median()

In [ ]:
df.loc[df['bill_length_mm'].isna()]

In [ ]:
df.bill_depth_mm = df.bill_depth_mm.fillna(
    df.groupby('species')['bill_depth_mm'].transform('median')
)
df.bill_length_mm = df.bill_length_mm.fillna(
    df.groupby('species')['bill_length_mm'].transform('median')
)
df.flipper_length_mm = df.flipper_length_mm.fillna(
    df.groupby('species')['flipper_length_mm'].transform('median')
)

Why `transform` here instead of just `groupby().mean()`? Because `transform` returns a Series aligned back to the original index/shape, so you can feed it straight into `fillna()` row-by-row — each missing value gets its own group's mean instead of a single overall number.

Try it on one column first, check with `.isna().sum()` before and after to confirm it worked, then think about whether you want to loop this over all the numeric measurement columns.

# Handle missing sex
Rows with a missing sex value can't be reliably guessed — drop only those rows, and report how many were removed.

In [ ]:
missing_sex_count = df.loc[df.sex.isna()].sex.size
df.dropna(subset='sex',inplace=True)
missing_sex_count

# Fix inconsistent labels
Standardize the sex column so values are consistently 'male'/'female' lowercase, and fix any stray typos or casing issues in species or island.

PS: This step isn't nessary because all the data in the `sex` column is already lowercase

In [ ]:
df['sex'] = df['sex'].str.lower()
df['sex']

# Flag outliers
Add a boolean column is_outlier that flags rows where `body_mass_g` is more than `3 standard deviations` from its species' `mean`.

PS: Think about what the actual `standard deviations from the mean` rule needs:

something to subtract `mean`

something to measure distance regardless of direction `absolute value`

then compare that distance to `3 * std`

In [ ]:
df['Flag_outliners'] = (df['body_mass_g'] - df['body_mass_g'].mean()).abs() > df['body_mass_g'].std() * 3
df.Flag_outliners.value_counts()

# Fix data types
Convert year to a proper integer type and species/island/sex to category dtype to save memory.

In [ ]:
df['year'].astype('int64')
df[['island', 'sex', 'species']].astype('category')
df.dtypes

# Create a size index
Add a column body_mass_kg converting grams to kilograms, and a bill_ratio column = bill_length_mm / bill_depth_mm.

In [ ]:
df['body_mass_kg'] = df.body_mass_g / 1000
df['bill_ratio'] = df.bill_length_mm / df.bill_depth_mm
df.head()

# Bucket into size classes
Add a size_class column: 'small', 'medium', or 'large' based on body_mass_g terciles (roughly equal-sized groups).

In [ ]:
df['size_class'] = pd.qcut(df.body_mass_g,3,labels=['Small','Medium','Large'])
df.head()

# Rename for the report
Rename columns to report-friendly labels, e.g. bill_length_mm → Bill Length (mm).

In [ ]:
df.rename(columns={ 'bill_length_mm':'Bill Length (mm)',
                    'bill_depth_mm':'Bill Depth (mm)',
                    'flipper_length_mm':'Flipper Length (mm)',
                    'body_mass_g':'Body Mass (g)',
                    'species':'Species'},
                    inplace=True)
df['sex'] = df.sex.str.title()
df.head()

# Species summary table
Build a summary table with one row per species showing count, mean body mass, mean flipper length, and mean bill length.x

In [ ]:
spec_summ = df.groupby('Species').agg({ 'Species':'count',
                                            'Body Mass (g)':'mean',
                                            'Bill Length (mm)':'mean'})
spec_summ.rename(columns={'Species':'Count'},inplace=True)
spec_summ

# Species x island breakdown
Build a table showing average body mass for every species/island combination that actually occurs in the data.

In [ ]:
island_summ = df.groupby(['Species','island'])['Body Mass (g)'].mean().unstack(fill_value=0)
island_summ

In [ ]:
df.pivot_table(values='Body Mass (g)',index='Species',columns='island',aggfunc='mean',fill_value=0)

# Sex comparison
For each species, compare average body mass between male and female penguins side by side.

In [ ]:
df.groupby(['Species','sex'])['Body Mass (g)'].mean().unstack()

In [ ]:
df.pivot_table('Body Mass (g)','Species','sex','mean')

# Year-over-year counts
Count how many penguins of each species were recorded per year, as a wide table (years as columns).

In [ ]:
over_year_count = df.groupby(['Species','year'])['sex'].count().unstack()

In [ ]:
df.pivot_table(values='sex',index='Species',columns='year',aggfunc='count')

# Top/bottom records
Find the 3 heaviest and 3 lightest penguins overall, including their species and island.

In [ ]:
body_mass_sorted = df.sort_values(ascending=False,by='Body Mass (g)')
top_bot_3 = pd.concat([body_mass_sorted.head(3),body_mass_sorted.tail(3)])
top_bot_3

# Build the summary export
Combine your species summary and species/island breakdown into one tidy DataFrame ready for export.

In [ ]:
penguins_summary = pd.concat([spec_summ, island_summ], axis=1)

# Export cleaned data
Save the cleaned, transformed DataFrame to penguins_clean.csv, and the summary table to penguins_summary.csv.

In [ ]:
df.to_csv('data/penguins_cleaned.csv')
penguins_summary.to_csv('data/penguins_summary.csv')

# Sanity check
Re-load both CSVs and confirm row counts and column dtypes match what you expect before sign-off.

In [ ]:
df = pd.read_csv('data/penguins.csv')
df.info()

In [ ]:
df_cleaned =pd.read_csv('data/penguins_cleaned.csv')
df_cleaned.info()